# 02 — Data preparation

**Goal.** Build a clean table with one row per patient-hour, split the **patients** into
train / calibration / test sets, and build the features of the baseline, with checks against
data leakage.

**Input.** The raw PhysioNet files `data/raw/training/training_setA/*.psv` and
`training_setB/*.psv` (see `data/README.md`).

**Outputs** (in `data/processed/`, not stored in git):
- `hourly.pkl` — one row per patient-hour, both hospitals
- `patients_splits.csv` — one row per patient, with its split
- `features_baseline.pkl` — features of the baseline (last observed values + "never measured" indicators)

**Run time.** About 10 minutes the first time (reading 40 336 files), then about a minute:
the hourly table is saved once and read back afterwards (cache).

## 0. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42                                   # same random choices on every computer
RAW = Path("../data/raw/training")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

## 1. Load all patients

Each `.psv` file is one patient: one row per hour in the ICU, 41 columns separated by `|`.

In [2]:
# List the files of each hospital (sorted: same order on every computer)
files_A = sorted((RAW / "training_setA").glob("*.psv"))
files_B = sorted((RAW / "training_setB").glob("*.psv"))
len(files_A), len(files_B)

(20336, 20000)

In [3]:
# Read one patient and add two columns: who the patient is and which hospital he comes from.
# This information is only in the file name and folder, so it would be lost when all patients are stacked.
def read_patient(path, hospital):
    df = pd.read_csv(path, sep="|")
    df["patient_id"] = path.stem     # file name without ".psv", e.g. "p000001"
    df["hospital"] = hospital        # "A" or "B"
    return df

read_patient(files_A[0], "A").head()

,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,patient_id,hospital
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,1,0,p000001,A
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,2,0,p000001,A
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,3,0,p000001,A
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,4,0,p000001,A
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,5,0,p000001,A


In [4]:
# Stack all patients in one table, or read the saved table if it already exists (cache).
# Delete hourly.pkl to force a new reading of the raw files.
HOURLY = PROCESSED / "hourly.pkl"

def read_hospital(files, hospital, chunk=2000):
    # Files are read 2 000 at a time and each batch is stacked at once: keeping 40 000 small
    # tables in memory before stacking them would need more than 4 GB of RAM.
    parts = []
    for start in range(0, len(files), chunk):
        frames = [read_patient(f, hospital) for f in files[start:start + chunk]]
        parts.append(pd.concat(frames, ignore_index=True))
    return pd.concat(parts, ignore_index=True)

if HOURLY.exists():
    data = pd.read_pickle(HOURLY)                    # a few seconds
else:
    data = pd.concat([read_hospital(files_A, "A"), read_hospital(files_B, "B")], ignore_index=True)
    data.to_pickle(HOURLY)                           # ~10 minutes, only once
data.shape

(1552210, 43)

In [5]:
# Check that no patient was lost
print("patients:", data["patient_id"].nunique())
print(data.groupby("hospital")["patient_id"].nunique())

patients: 40336
hospital
A    20336
B    20000
Name: patient_id, dtype: int64


**Result.** 1 552 210 patient-hours, 43 columns (41 + `patient_id` + `hospital`);
40 336 patients: 20 336 in hospital A and 20 000 in hospital B, as announced by PhysioNet.

## 2. Quality checks

In [6]:
# Check 1: hours follow each other.
# diff() = gap with the previous row; groupby("patient_id") does it patient by patient,
# so the last hour of one patient is never compared with the first hour of the next one.
step = data.groupby("patient_id")["ICULOS"].diff()
step.value_counts(dropna=False)

ICULOS
1.0    1511874
NaN      40336
Name: count, dtype: int64

In [7]:
# Check 2: once a patient is labelled sepsis (1), he stays at 1 until the end of the stay.
# A negative gap would mean a 1 -> 0 change, i.e. an error in the labels.
label_step = data.groupby("patient_id")["SepsisLabel"].diff()
(label_step < 0).sum()

np.int64(0)

**Result.**
- ICULOS: 1 511 874 gaps of exactly 1 hour, plus 40 336 `NaN` (the first hour of each patient has no
  previous row). No missing or duplicated hour: one row = one hour, in time order.
- SepsisLabel never goes back from 1 to 0.

The data is clean on these two points; nothing needs to be corrected.

## 3. Patient-level split (60 / 20 / 20 in each hospital)

We split **patients**, not rows. If rows were split at random, hours 1–30 of a patient could be in
the train set and hours 31–50 in the test set: the model would already have seen this patient and
its score would be too optimistic (data leakage). Here, all hours of a patient go to the same set.

In [8]:
# One row per patient
patients = data.groupby("patient_id").agg(
    hospital=("hospital", "first"),
    n_hours=("ICULOS", "size"),             # length of stay (hours)
    ever_sepsis=("SepsisLabel", "max"),     # 1 if at least one hour is labelled sepsis
).reset_index()
patients.head()

,patient_id,hospital,n_hours,ever_sepsis
0,p000001,A,54,0
1,p000002,A,23,0
2,p000003,A,48,0
3,p000004,A,29,0
4,p000005,A,48,0


In [9]:
# Number and share of septic patients per hospital
patients.groupby("hospital")["ever_sepsis"].agg(["size", "sum", "mean"])

,size,sum,mean
hospital,,,
A,20336,1790,0.088021
B,20000,1142,0.057100


In [10]:
# Two cuts: 60 % train, then the other 40 % in two halves -> 20 % calibration, 20 % test.
# stratify = same share of septic patients in the three sets.
# Both hospitals are split, so the experiment can be run in both directions (A -> B and B -> A).
def split_hospital(p):
    train, rest = train_test_split(p, train_size=0.6, stratify=p["ever_sepsis"], random_state=SEED)
    calib, test = train_test_split(rest, train_size=0.5, stratify=rest["ever_sepsis"], random_state=SEED)
    return pd.concat([train.assign(split="train"), calib.assign(split="calib"), test.assign(split="test")])

patients = pd.concat([split_hospital(p) for _, p in patients.groupby("hospital")], ignore_index=True)
patients.groupby(["hospital", "split"])["ever_sepsis"].agg(["size", "mean"])

size      mean
hospital split                 
A        calib   4067  0.088026
         test    4068  0.088004
         train  12201  0.088026
B        calib   4000  0.057250
         test    4000  0.057000
         train  12000  0.057083

In [11]:
# Each patient appears once, so he is in exactly one set (no patient in both train and test)
print("each patient in one set only:", patients["patient_id"].is_unique)
patients.to_csv(PROCESSED / "patients_splits.csv", index=False)

each patient in one set only: True


**Result.**
- Hospital A: 12 201 train / 4 067 calibration / 4 068 test patients. Hospital B: 12 000 / 4 000 / 4 000.
- 8.8 % of patients develop sepsis in A (1 790 patients) vs 5.7 % in B (1 142): sepsis is about
  1.5 times more frequent in A (**label-prevalence shift** between the hospitals).
- Thanks to the stratification, this share is the same in the three sets of each hospital.

## 4. Baseline features

**Rule:** at hour t, the model may only use what was known at hour t.

- Each empty cell is filled with the **last observed value of the same patient** (LOCF: last
  observation carried forward) — never with a value of another patient, never with a future value.
- After this filling, an empty cell means "never measured in this patient up to this hour".
  We turn it into a 0/1 indicator: not having ordered a lab test is itself clinical information.

In [12]:
# The 40 input variables, and the 3 that are never missing (they get no indicator)
VARIABLES = [c for c in data.columns if c not in ["SepsisLabel", "patient_id", "hospital"]]
ALWAYS_PRESENT = ["Age", "Gender", "ICULOS"]
print(len(VARIABLES), "variables")
print(data[ALWAYS_PRESENT].isna().sum())

40 variables
Age       0
Gender    0
ICULOS    0
dtype: int64


In [13]:
# Fill with the last observed value, patient by patient.
# This only works because rows are in time order (checked in section 2).
locf = data.groupby("patient_id")[VARIABLES].ffill()

# Example on the first patient: Temp is measured only every few hours
pd.DataFrame({"Temp_raw": data["Temp"], "Temp_filled": locf["Temp"]}).head(10)

,Temp_raw,Temp_filled
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
5,NaN,NaN
6,36.11,36.11
7,NaN,36.11
8,NaN,36.11
9,NaN,36.11


In [14]:
# "Never measured so far" indicators (1 = never measured in this patient up to this hour)
to_flag = [v for v in VARIABLES if v not in ALWAYS_PRESENT]
missing = locf[to_flag].isna().astype(int).add_suffix("_missing")
missing.mean().sort_values(ascending=False).head(10)

Bilirubin_direct_missing    0.953425
EtCO2_missing               0.915879
Fibrinogen_missing          0.892848
TroponinI_missing           0.849468
Lactate_missing             0.701823
SaO2_missing                0.698076
Bilirubin_total_missing     0.697902
Alkalinephos_missing        0.696910
AST_missing                 0.694009
BaseExcess_missing          0.681044
dtype: float64

In [15]:
# Assemble the feature table and save it
features = pd.concat([data[["patient_id", "hospital", "SepsisLabel"]], locf, missing], axis=1)
print(features.shape)
features.to_pickle(PROCESSED / "features_baseline.pkl")

(1552210, 80)


**Result.** 1 552 210 rows and 80 columns: 3 identification columns, 40 filled variables and
37 indicators. Some variables are never measured in most hours: Bilirubin_direct in 95 % of hours,
EtCO2 in 92 %, Fibrinogen in 89 %, TroponinI in 85 %, Lactate in 70 %.

**Not done here on purpose:** the cells that stay empty (variable never measured yet) are **not**
filled with a median. A median is learned from data; computed on the whole table, it would contain
information from the test patients (leakage). It is done in the Pipeline of the baseline
(notebook 04), fitted on the training patients only.

## 5. Leakage test

**Idea.** Take a patient as he was at hour t: delete all his hours after t and recompute his
features. If they are identical to the features computed with the full stay, they do not use
the future.

In [16]:
# Same computation as section 4, in a function, so it can be applied to a cut patient
def build_features(df):
    locf = df.groupby("patient_id")[VARIABLES].ffill()
    missing = locf[to_flag].isna().astype(int).add_suffix("_missing")
    return pd.concat([df[["patient_id", "hospital", "SepsisLabel"]], locf, missing], axis=1)

# assert_frame_equal says nothing if the two tables are identical, and raises an error otherwise
pd.testing.assert_frame_equal(build_features(data), features)
print("The function gives exactly the table of section 4")

The function gives exactly the table of section 4


In [17]:
# Test on 50 random patients, each cut at a random hour
rng = np.random.default_rng(SEED)

for pid in rng.choice(data["patient_id"].unique(), size=50, replace=False):
    one = data[data["patient_id"] == pid]          # full stay of the patient
    cut = rng.integers(1, len(one) + 1)            # keep the first `cut` hours
    truncated = one.iloc[:cut]                     # what was known at that moment
    pd.testing.assert_frame_equal(build_features(truncated), features.loc[truncated.index])

print("Leakage test passed: for 50 patients, no feature uses the future")

Leakage test passed: for 50 patients, no feature uses the future


In [18]:
# A test that can never fail proves nothing: we run it on a deliberately WRONG version
# that fills empty cells with the NEXT value (bfill = backward fill), i.e. with the future.
def build_features_leaky(df):
    return df.groupby("patient_id")[VARIABLES].bfill()   # wrong on purpose: uses the future

n_detected = 0
for pid in rng.choice(data["patient_id"].unique(), size=50, replace=False):
    one = data[data["patient_id"] == pid]
    cut = rng.integers(1, len(one) + 1)
    # .equals returns True/False instead of raising an error, so we can count
    if not build_features_leaky(one.iloc[:cut]).equals(build_features_leaky(one).iloc[:cut]):
        n_detected += 1

print(f"Leak detected for {n_detected} patients out of 50")

Leak detected for 43 patients out of 50


**Result.**
- Our features pass the test for the 50 patients: they only use the past.
- The wrong version (bfill) is caught for 43 patients out of 50, so the test is able to detect a leak.
  The 7 others are cases where nothing is measured after the cut, so cutting or not changes nothing,
  even with bfill.